In [1]:
import cv2
import numpy as np 


### dense optical flow 


In [2]:
# Function to track moving cars using dense optical flow
def track_cars_dense(video_path):
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Create VideoWriter object to save the output video
    out = cv2.VideoWriter('car_tracking_dense_flow.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Read the first frame
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        # Read the current frame
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the current frame to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Get the magnitude and angle of the flow vectors
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Threshold for motion detection
        thresh = 5
        motion_mask = mag > thresh

        # Apply the motion mask to the current frame
        result_frame = frame.copy()
        result_frame[motion_mask] = [0, 0, 255]  # Highlight moving objects in red

        # Show the result on the frame
        cv2.imshow('Result', result_frame)
        out.write(result_frame)

        # Update the previous frame
        prev_gray = gray

        # Break the loop if 'q' is pressed
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()

# Replace 'your_video_path.mp4' with the actual path to your video file
video_path = 'car_-_2165 (540p).mp4'
track_cars_dense(video_path) 


libGL error: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: iris
libGL error: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: iris
libGL error: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: swrast


### sparse optical flow 


In [3]:
# Function to track moving cars using sparse optical flow
def track_cars_sparse(video_path):
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Create VideoWriter object to save the output video
    out = cv2.VideoWriter('car_tracking_sparse_flow.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Parameters for Shi-Tomasi corner detection
    feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

    # Parameters for Lucas-Kanade optical flow
    lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    # Read the first frame
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    # Detect feature points using Shi-Tomasi corner detection
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

    # Create a mask for drawing tracks
    mask = np.zeros_like(prev_frame)

    # Create some random colors for the tracks
    color = np.random.randint(0, 255, (100, 3))

    while True:
        # Read the current frame
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the current frame to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate optical flow using Lucas-Kanade method
        new_pts, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **lk_params)

        # Check if new_pts is not None and status is not None
        if new_pts is not None and status is not None:
            # Select good points
            good_new = new_pts[status == 1]
            good_prev = prev_pts[status == 1]

            # Draw tracks on the mask
            for i, (new, prev) in enumerate(zip(good_new, good_prev)):
                a, b = new.ravel()
                c, d = prev.ravel()
                mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
                frame = cv2.circle(frame, (int(a), int(b)), 5, color[i].tolist(), -1)

            # Combine the frame with the mask
            result_frame = cv2.add(frame, mask)

            # Show the result on the frame
            cv2.imshow('Sparse Optical Flow Result', result_frame)
            out.write(result_frame)

            # Update the previous frame and points
            prev_gray = gray.copy()

            # Detect feature points using Shi-Tomasi corner detection in the current frame
            prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

        # Clear the mask for the next iteration
        mask = np.zeros_like(prev_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()

# Replace 'your_video_path.mp4' with the actual path to your video file
video_path = 'car_-_2165 (540p).mp4'
track_cars_sparse(video_path)


### combination of dense optical flow of background subtraction 


In [4]:
def track_cars_combined_dense(video_path):
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Create VideoWriter object to save the output video
    out = cv2.VideoWriter('car_tracking_background_dense.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Background subtractor using MOG2
    bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=False)

    # Read the first frame
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        # Read the current frame
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the current frame to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Get the magnitude and angle of the flow vectors
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Threshold for motion detection using dense optical flow
        flow_thresh = 5
        motion_mask_flow = (mag > flow_thresh).astype(np.uint8)  # Ensure the data type is uint8

        # Apply MOG2 background subtraction to the current frame
        fg_mask_mog2 = bg_subtractor.apply(frame)
        _, fg_mask_mog2 = cv2.threshold(fg_mask_mog2, 240, 255, cv2.THRESH_BINARY)

        # Combine the masks
        combined_mask = cv2.bitwise_or(fg_mask_mog2, motion_mask_flow)

        # Apply the combined mask to the current frame
        result_frame = frame.copy()
        result_frame[combined_mask != 0] = [0, 0, 255]  # Highlight moving objects in red

        # Show the result on the frame
        cv2.imshow('Result', result_frame)
        out.write(result_frame)

        # Update the previous frame
        prev_gray = gray

        # Break the loop if 'q' is pressed
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()

# Replace 'your_video_path.mp4' with the actual path to your video file
video_path = 'car_-_2165 (540p).mp4'
track_cars_combined_dense(video_path)
